# MongoDB Landing Zone — Exploration

Verbindung: `liora-vm.matthiaskoehler.com:27017`  
Datenbank: `airline_landing`  
Collection: `adsb_raw`  

Befüllt durch `collectors/adsb_collector.py` (adsb.lol, Berlin 60 nm Radius).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from datetime import timezone

sys.path.insert(0, '.')
from db.mongo.connector import from_env

db = from_env().connect()
print("Verbunden mit MongoDB")

## 1. Collections und Dokumentenanzahl

In [ ]:
col = db.collection('adsb_raw')

total = col.count_documents({})
print(f"Snapshots in adsb_raw: {total}")

## 2. Neueste Snapshots (ohne `ac`-Array)

In [ ]:
recent = list(col.find({}, {'ac': 0}).sort('collected_at', -1).limit(10))

df_snaps = pd.DataFrame([
    {
        'collected_at': d['collected_at'],
        'total': d['total'],
        'source': d['source'],
        '_id': str(d['_id']),
    }
    for d in recent
])
print(df_snaps.to_string(index=False))

## 3. Letzten Snapshot als DataFrame aufklappen

In [ ]:
latest = col.find_one({}, sort=[('collected_at', -1)])
print(f"Snapshot: {latest['collected_at']}  —  {latest['total']} aircraft")

df = pd.DataFrame(latest['ac'])
print(f"Shape: {df.shape}")
print(f"Spalten: {list(df.columns)}")

In [ ]:
# Lesbare Übersicht
cols = [c for c in ['hex', 'flight', 'r', 't', 'alt_baro', 'gs', 'lat', 'lon', 'dst'] if c in df.columns]
print(df[cols].to_string(index=False))

## 4. Datenqualität — Vollständigkeit der Kernfelder

In [ ]:
key_fields = ['hex', 'flight', 'lat', 'lon', 'alt_baro', 'gs', 'r', 't']
present = {f: df[f].notna().sum() if f in df.columns else 0 for f in key_fields}
for field, count in present.items():
    pct = count / len(df) * 100
    print(f"  {field:<12} {count:>3}/{len(df)}  ({pct:.0f}%)")

## 5. alt_baro: Zahl vs. 'ground'

In [ ]:
if 'alt_baro' in df.columns:
    on_ground = (df['alt_baro'] == 'ground').sum()
    airborne  = (df['alt_baro'] != 'ground').sum()
    print(f"Am Boden:    {on_ground}")
    print(f"In der Luft: {airborne}")

    df['alt_baro_ft'] = pd.to_numeric(df['alt_baro'], errors='coerce')
    df['on_ground']   = df['alt_baro'] == 'ground'

## 6. Top-Callsigns über Berlin

In [ ]:
if 'flight' in df.columns:
    top = (
        df['flight']
        .str.strip()
        .replace('', pd.NA)
        .dropna()
        .value_counts()
        .head(20)
    )
    print(top.to_string())

## 7. Alle Snapshots zusammenführen — Zeitreihe der Flugzeugzahl

In [ ]:
all_snaps = list(col.find({}, {'ac': 0}).sort('collected_at', 1))

df_ts = pd.DataFrame([
    {'collected_at': d['collected_at'], 'total': d['total']}
    for d in all_snaps
])
df_ts['collected_at'] = pd.to_datetime(df_ts['collected_at'], utc=True)

print(df_ts.to_string(index=False))
print(f"\nSchnitt: {df_ts['total'].mean():.1f} aircraft/snapshot")
print(f"Min/Max: {df_ts['total'].min()} / {df_ts['total'].max()}")

## 8. Verbindung schließen

In [ ]:
db.close()
print("Verbindung geschlossen")